# Dynamic Resilient Grid - analyst walkthrough

An end-to-end tour of the DRG framework: load the data, look at the load curves, train the
forecaster, detect statistical stress, run an electrification scenario, and read the SHAP
explanation.

Run `python -m drg.cli run-all` once before this notebook so the cached artifacts exist,
or let each cell build what it needs.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from drg.config import load_config

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

cfg = load_config(Path.cwd().parent / "configs" / "config.yaml")
cfg.paths.root

## 1. Data

The cleaned neighbourhood table: one row per (timestamp, neighbourhood).

In [ ]:
from drg.pipeline import stage_ingest

demand = stage_ingest(cfg)
print(f"{len(demand):,} rows | {demand['neighbourhood_id'].nunique()} neighbourhoods")
print(f"{demand['timestamp'].min()} -> {demand['timestamp'].max()}")
print(f"source: {demand['source'].iloc[0]}")
demand.head()

## 2. Load curves

Weekday against weekend, and the winter amplification the proposal asks about.

In [ ]:
d = demand.copy()
d["period"] = d["timestamp"].dt.hour * 2 + d["timestamp"].dt.minute // 30
d["day_type"] = np.where(d["timestamp"].dt.dayofweek >= 5, "weekend", "weekday")

fig, ax = plt.subplots(figsize=(11, 5))
for day_type, style in (("weekday", "-"), ("weekend", "--")):
    profile = d[d.day_type == day_type].groupby("period")["demand_kwh"].mean()
    ax.plot(profile.index / 2, profile.values, style, lw=2, label=day_type)
ax.axvspan(17, 22, alpha=0.12, color="tab:red", label="evening peak window")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Mean kWh per half hour")
ax.set_title("Neighbourhood load curve")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
month = demand["timestamp"].dt.month
winter = demand[month.isin([12, 1, 2])]["demand_kwh"].mean()
summer = demand[month.isin([6, 7, 8])]["demand_kwh"].mean()
print(f"winter mean {winter:.2f} kWh | summer mean {summer:.2f} kWh")
print(f"winter uplift: {100 * (winter / summer - 1):.1f}%")

## 3. Temperature response

The demand-temperature relationship is what makes heat-pump electrification bite. Temperature
here is observed ERA5 reanalysis pulled from the key-free Open-Meteo archive.

In [ ]:
if "temperature_c" in demand.columns:
    d2 = demand.dropna(subset=["temperature_c"]).copy()
    d2["bin"] = pd.cut(d2["temperature_c"], bins=np.arange(-6, 32, 2))
    resp = d2.groupby("bin", observed=True)["demand_kwh"].mean()
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot([i.mid for i in resp.index], resp.values, "o-", lw=2)
    ax.set_xlabel("Air temperature (deg C)")
    ax.set_ylabel("Mean kWh per half hour")
    ax.set_title("Temperature response of neighbourhood demand")
    ax.grid(alpha=0.3)
    plt.show()

## 4. Features and forecasting

Chronological split, then the full model ladder. The naive benchmark is what the headline
accuracy has to beat to mean anything.

In [ ]:
from drg.pipeline import stage_features

features = stage_features(cfg)
print(f"{len(features):,} rows x {features.shape[1]} columns")
[c for c in features.columns if c.startswith(("lag_", "roll_", "ramp"))][:12]

In [ ]:
from drg.models.train import train_models

result = train_models(features, cfg, train_lstm=False, save=False)
result.metrics[["model", "mae", "rmse", "mape", "r2", "stress_f1"]]

In [ ]:
preds = result.predictions.sort_values("timestamp")
site = preds["neighbourhood_id"].iloc[0]
window = preds[preds.neighbourhood_id == site].tail(336)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(window["timestamp"], window["actual"], lw=2, label="actual")
ax.plot(window["timestamp"], window["xgboost"], lw=2, ls="--", label="XGBoost forecast")
ax.set_ylabel("kWh per half hour")
ax.set_title(f"One week of test-window forecasts ({site})")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 5. Statistical stress

The 95th percentile of each site own history, held fixed for every later scenario.

In [ ]:
from drg.stress.detection import compute_thresholds, detect_stress, stress_events, stress_summary

thresholds = compute_thresholds(
    demand,
    percentile=cfg.stress["primary_percentile"],
    sigma=cfg.stress["sensitivity_sigma"],
)
flagged = detect_stress(demand, thresholds)
events = stress_events(flagged, min_periods=cfg.stress["min_event_periods"])
stress_summary(flagged, events)[
    ["neighbourhood_id", "threshold_kwh", "stress_frequency_pct", "n_events", "mean_event_hours"]
]

In [ ]:
from drg.stress.detection import diurnal_stress_profile

diurnal = diurnal_stress_profile(flagged)
diurnal = diurnal[diurnal.neighbourhood_id == site]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(diurnal["period_of_day"] / 2, diurnal["stress_rate_pct"], width=0.45)
ax.set_xlabel("Hour of day")
ax.set_ylabel("% of half-hours in stress")
ax.set_title(f"When stress occurs ({site})")
ax.grid(alpha=0.3, axis="y")
plt.show()

## 6. Electrification scenario

Bottom-up EV and heat-pump simulation. Check the reported ADMD against published UK DNO figures
(roughly 1.5-2 kW per technology) before trusting the headline.

In [ ]:
from drg.simulation.electrification import EVConfig, HeatPumpConfig, apply_scenario

window_df = demand[demand.timestamp >= demand.timestamp.max() - pd.Timedelta(days=180)]
scenario = apply_scenario(
    window_df,
    ev_adoption=0.6,
    hp_adoption=0.5,
    ev_config=EVConfig.from_config(cfg.electrification["ev"]),
    hp_config=HeatPumpConfig.from_config(cfg.electrification["heat_pump"]),
    seed=cfg.seed,
    runs=5,
)
pd.Series(scenario.summary)

In [ ]:
profile = (
    scenario.frame.assign(period=lambda x: x.timestamp.dt.hour * 2 + x.timestamp.dt.minute // 30)
    .groupby("period")[["base_kwh", "ev_kwh", "heat_pump_kwh"]]
    .mean()
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(
    profile.index / 2,
    profile["base_kwh"], profile["ev_kwh"], profile["heat_pump_kwh"],
    labels=["base demand", "EV charging", "heat pumps"],
    alpha=0.85,
)
ax.axhline(thresholds[site].primary_kwh, ls="--", color="0.35", label="P95 stress threshold")
ax.set_xlabel("Hour of day")
ax.set_ylabel("kWh per half hour")
ax.set_title("Mean daily profile: EV 60%, heat pumps 50%")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 7. Sensitivity grid

How peak amplification and stress escalate across the adoption space.

In [ ]:
from drg.analysis.sensitivity import run_sensitivity_grid, summarise_amplification

grid, _ = run_sensitivity_grid(
    window_df,
    thresholds,
    ev_levels=cfg.electrification["ev"]["adoption_levels"],
    hp_levels=cfg.electrification["heat_pump"]["adoption_levels"],
    ev_config=EVConfig.from_config(cfg.electrification["ev"]),
    hp_config=HeatPumpConfig.from_config(cfg.electrification["heat_pump"]),
    seed=cfg.seed,
    runs=1,
)
grid.pivot_table(index="hp_adoption", columns="ev_adoption", values="peak_amplification_pct").round(1)

In [ ]:
pd.Series(summarise_amplification(grid))

## 8. Explainability

Global drivers, and the drivers that matter specifically during stress periods - which is the
view a network planner actually needs.

In [ ]:
from drg.explain.shap_explain import explain_model

test = features[features.timestamp >= features.timestamp.max() - pd.Timedelta(days=30)]
threshold = np.percentile(features["target"], cfg.stress["primary_percentile"])

report = explain_model(
    result.bundles["xgboost"].estimator,
    test[result.best.feature_columns],
    stress_mask=(test["target"] >= threshold).to_numpy(),
    sample_size=2000,
    seed=cfg.seed,
)
report.global_importance.head(12)

In [ ]:
if len(report.stress_importance):
    comparison = (
        report.global_importance[["feature", "contribution_pct"]]
        .merge(
            report.stress_importance[["feature", "contribution_pct"]],
            on="feature",
            suffixes=("_all", "_stress"),
        )
        .head(10)
    )
    display(comparison)

In [ ]:
report.explain_row(0)

## 9. Streaming replay

The same engine that drives the dashboard and the API: a rolling forecast plus a pre-emptive
alert when the forecast breaches the threshold before the measurement does.

In [ ]:
from drg.streaming.replay import StreamingReplayEngine

engine = StreamingReplayEngine(
    demand,
    result.bundles["xgboost"],
    thresholds,
    cfg,
    neighbourhood_id=site,
    start_index=len(demand[demand.neighbourhood_id == site]) - 200,
)
ticks = engine.run(n=96)
ticks[["timestamp", "actual_kwh", "forecast_next_kwh", "threshold_kwh", "severity"]].tail(10)